In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import re
import plotly.io as pio
url = "https://ldcom365.sharepoint.com"



In [0]:

history=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/merged/history and lineups.xlsx')

ref_date = datetime.today() - timedelta(days=5)

# Step 2: Extract year and month of that reference date
ref_year = ref_date.year
ref_month = ref_date.month


mask_next = history['Flag'].astype(str).str.lower() == 'next'

# Step 2: Move the Date for 'Next' flags
def move_to_next_month(date):
    if date.month == 12:
        return pd.Timestamp(year=date.year + 1, month=1, day=5)
    else:
        return pd.Timestamp(year=date.year, month=date.month + 1, day=5)

history.loc[mask_next, 'Date'] = history.loc[mask_next, 'Date'].apply(move_to_next_month)
history.loc[mask_next, 'Flag'] = ''

# Step 3: Update Year and Month
history.loc[mask_next, 'Year'] = history.loc[mask_next, 'Date'].dt.year
history.loc[mask_next, 'Month'] = history.loc[mask_next, 'Date'].dt.month

# Step 4: Clear 'to be reviewed' or 'to be checked' Flags
mask_clear_flag = history['Flag'].astype(str).str.lower().isin(['to be reviewed', 'to be checked',])

mask_clear_flag_status = history['Flag'].astype(str).str.lower().isin(['alongside', 'at roads','at san nicolas','at rosario'])

history.loc[mask_clear_flag_status, 'Flag'] = ''
current_28th = datetime.today().replace(day=28)

# Assign it to the 'Date' column for the selected rows
history.loc[mask_clear_flag_status, 'Date'] = current_28th
history.loc[mask_clear_flag_status, 'Month'] = history.loc[mask_clear_flag_status, 'Date'].dt.month
history.loc[mask_clear_flag_status, 'Year'] = history.loc[mask_clear_flag_status, 'Date'].dt.year

mask_unknown = history['Destination'].isna() & history['Region'].isna()
history.loc[mask_unknown, ['Destination', 'Region']] = 'UNKNOWN'

history.loc[mask_clear_flag, 'Flag'] = ''


In [0]:
# sp_mgr.rm('/sites/GRP-TradingLineups/Lineups/merged/history and lineups.xlsx')
# sp_mgr.save_pd_to_excel('/sites/GRP-TradingLineups/Lineups/merged/history and lineups.xlsx',history,index=False)

## CURRENT MONTH

In [0]:
# Step 3: Filter the DataFrame
filtered_df = history[(history['Date'].dt.year == ref_year) & (history['Date'].dt.month == ref_month)]

filtered_df_SBM=filtered_df[filtered_df['Product']=='SBO']
filtered_df_SBM['Date'] = filtered_df_SBM['Date'].dt.date

### BY REGION

In [0]:
# Group sailed quantities by region
sailed = filtered_df_SBM[filtered_df_SBM['Status'] == 'SAILED'].groupby('Region', as_index=False)['Quantity'].sum()
sailed = sailed.rename(columns={'Quantity': 'Quantity_Sailed'})

# Group announced quantities by region
announced = filtered_df_SBM[filtered_df_SBM['Status'] == 'ANNOUNCED'].groupby('Region', as_index=False)['Quantity'].sum()
announced = announced.rename(columns={'Quantity': 'Quantity_Announced'})

# Merge both
region_totals = pd.merge(sailed, announced, on='Region', how='outer').fillna(0)

# Add total column
region_totals['Total'] = region_totals['Quantity_Sailed'] + region_totals['Quantity_Announced']

# Optional: sort by total
region_totals = region_totals.sort_values(by='Total', ascending=False)

filtered_df_year = history[(history['Date'].dt.year == ref_year)]
filtered_df_SBM_year=filtered_df_year[filtered_df_year['Product']=='SBM']
filtered_df_SBM_year=filtered_df_SBM_year[filtered_df_SBM_year['Origin']=='ARG']


# Ensure Date is datetime type if not already
filtered_df_SBM_year['Date'] = pd.to_datetime(filtered_df_SBM_year['Date'])

# Add Month column (numerical)
filtered_df_SBM_year['Month'] = filtered_df_SBM_year['Date'].dt.month

# Clean country names if needed
filtered_df_SBM_year['Destination'] = filtered_df_SBM_year['Destination'].str.upper().str.strip()

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table = (
    filtered_df_SBM_year
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
    .sort_index(axis=1)     # Ensure months go 1 to 12
)

# Optional: round values
monthly_table = monthly_table.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table.columns.name = None
monthly_table = monthly_table.reset_index()


# Clean country names if needed
filtered_df_SBM_year['Destination'] = filtered_df_SBM_year['Destination'].str.upper().str.strip()

filtered_df_SBM_year_sailed=filtered_df_SBM_year[filtered_df_SBM_year['Status']=='SAILED']
filtered_df_SBM_year_anc=filtered_df_SBM_year[filtered_df_SBM_year['Status']=='ANNOUNCED']

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table_sailed = (
    filtered_df_SBM_year_sailed
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
    .sort_index(axis=1)     # Ensure months go 1 to 12
)

# Optional: round values
monthly_table_sailed = monthly_table_sailed.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table_sailed.columns.name = None
monthly_table_sailed = monthly_table_sailed.reset_index()

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table_ancd = (
    filtered_df_SBM_year_anc
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
    .sort_index(axis=1)     # Ensure months go 1 to 12
)

# Optional: round values
monthly_table_ancd = monthly_table_ancd.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table_ancd.columns.name = None
monthly_table_ancd = monthly_table_ancd.reset_index()

from datetime import datetime

# Step 1: Add 'Month' column if missing
filtered_df_SBM_year['Month'] = filtered_df_SBM_year['Date'].dt.month

# Step 2: Get current month + 1
cutoff_month = (datetime.today().month + 1)

# Step 3: Filter up to current month + 1
filtered_df_SBM_cut = filtered_df_SBM_year[filtered_df_SBM_year['Month'] <= cutoff_month]

# Step 4: Create pivot table by Region and Month
quantity_by_region_month = pd.pivot_table(
    filtered_df_SBM_cut,
    values='Quantity',
    index='Region',
    columns='Month',
    aggfunc='sum',
    fill_value=0
)

# Step 5: Replace month numbers with names (Jan, Feb, etc.)
quantity_by_region_month.columns = [datetime(1900, m, 1).strftime('%b') for m in quantity_by_region_month.columns]

ref_date = datetime.today() - timedelta(days=5)

# Step 2: Extract year and month of that reference date
ref_year = ref_date.year
ref_month = ref_date.month

history_SBM=history[history['Product']=='SBM']

# Step 3: Filter the DataFrame
history_SBM = history_SBM[(history_SBM['Date'].dt.month == ref_month)]

sailed_df = history_SBM[history_SBM['Status'] == 'SAILED'].copy()

# Step 2: Make sure 'Date' is a datetime column
sailed_df['Date'] = pd.to_datetime(sailed_df['Date'])

# Step 3: Sort by 'Date'
sailed_df = sailed_df.sort_values(by='Date')

# Step 4: Compute the cumulative sum of 'Quantity'
sailed_df['Sailed'] = sailed_df['Quantity'].cumsum()



### OSA

In [0]:
country_mapping={"S. ARABIA":"SAUDI ARABIA","SAUDI ARABI":"S. ARABIA"}


country_region_mapping = {
    "ALGERIA": "AFRICA",
    'ALEGERIA':'AFRICA',
    "ARGENTINA": "SOUTH AMERICA",
    "BANGLADESH": "ASIA",
    "BELGIUM": "EU",
    "BRAZIL": "SOUTH AMERICA",
    "CHILE": "SOUTH AMERICA",
    "CHINA": "ASIA",
    "COLOMBIA": "SOUTH AMERICA",
    "COSTA RICA": "CENTRAL AMERICA",
    "CUBA": "CENTRAL AMERICA",
    "CYPRUS": "EU",
    "CYPRUS/GREECE": "EU",
    "DENMARK": "EU",
    "EGYPT": "AFRICA",
    "FRANCE": "EU",
    "GERMANY": "EU",
    "GREECE": "EU",
    "EU-28":"EU",
    "GREECE/ISRAEL": "EU",
    "GREECE/ITALY": "EU",
    "HOLLAND": "EU",
    "HOLLAND/GERMANY": "EU",
    "HONDURAS": "CENTRAL AMERICA",
    "INDONESIA": "SE ASIA",
    "IRAN": "ME ASIA",
    "IRELAND": "EU",
    "ISRAEL": "ME ASIA",
    "ISRAEL/GREECE": "EU",
    "ITALY": "EU",
    "IVORY COAST": "AFRICA",
    "JAPAN": "ASIA",
    "JORDAN": "ME ASIA",
    "KENYA": "AFRICA",
    "KOREA": "ASIA",
    "LEBANON": "ME ASIA",
    "LITHUANIA": "EU",
    "MALAYSIA": "SE ASIA",
    "MAURITIUS": "AFRICA",
    "MEXICO": "CENTRAL AMERICA",
    "MOROCCO": "AFRICA",
    "NETHERLANDS": "EU",
    "NETHERLANDS/GERMANY": "EU",
    "NIGERIA": "AFRICA",
    "PANAMA": "CENTRAL AMERICA",
    "PERU": "SOUTH AMERICA",
    "PHILIPPINES": "SE ASIA",
    "PORTUGAL": "EU",
    "PUERTO RICO": "CENTRAL AMERICA",
    "ROMANIA": "EU",
    "RUSSIA": "ASIA",
    "SAF": "AFRICA",
    "SAUDI ARABIA": "ME ASIA",
    "SENEGAL": "AFRICA",
    "SOUTH AFRICA": "AFRICA",
    "SOUTH KOREA": "ASIA",
    "SPAIN": "EU",
    "SYRIA": "ME ASIA",
    "TAIWAN": "ASIA",
    "TBC": "TBC",
    "THAILAND": "SE ASIA",
    "TUNISIA": "AFRICA",
    "TURKEY": "ME ASIA",
    "U.ARAB EMIRAT": "ME ASIA",
    "UAE": "ME ASIA",
    "UK": "EU",
    "UNITED KINGDOM": "EU",
    "POLAND": "EU",
    "SLOVENIA": "EU",
    "LITUANIA": "EU",
    "CROATIA": "EU",
    "GREECE + CYPRUS":'EU',

    "UNITED ARAB EMIRATES": "ME ASIA",
    "URUGUAY": "SOUTH AMERICA",
    "USA": "NORTH AMERICA",
    "UNITED STATES": "NORTH AMERICA",
    "VENEZUELA": "SOUTH AMERICA",
    "VIETNAM": "SE ASIA",
    "YEMEN": "ME ASIA",
    "": "NOT AVAILABLE",
    "MOZAMBIQUE": "AFRICA",
    "GUATEMALA": "CENTRAL AMERICA",
    "US": "NORTH AMERICA",
    "Z. OTHER EAST AFRICA": "AFRICA",
    "DOMINICAN REPUBLIC": "CENTRAL AMERICA",
    "AUSTRALIA": "OCEANIA",
    "NEW ZEALAND": "OCEANIA",
    "EL SALVADOR": "CENTRAL AMERICA",
    "NICARAGUA": "CENTRAL AMERICA",
    "OMAN": "ME ASIA",
    "KUWAIT": "ME ASIA",
    "SWITZERLAND": "EU",
    "IRAQ": "ME ASIA",
    "HAITI": "CENTRAL AMERICA",
    "CANADA": "NORTH AMERICA",
    "TRINIDAD": "CENTRAL AMERICA",
    "JAMAICA": "CENTRAL AMERICA",
    "ANGOLA": "AFRICA",
    "Z. OTHER FSU": "EUROPE",
    "NORWAY": "EU",
    "NOT AVAILABLE": "",
    "ECUADOR":"SOUTH AMERICA",
    'GUYANA':"SOUTH AMERICA",
    'TANZANIA':'AFRICA',
    'LUANDA':'AFRICA',
    'LIBYA':'AFRICA', 
    'PHILIPPINNES':'SE ASIA',
    'REUNION ISLAND':'AFRICA',
    'GHANA':'AFRICA',
    'CONGO':'AFRICA',
    'NAMIBIA':'AFRICA',
    'CAPE VERDE':'AFRICA',
    'MAURITANA':'AFRICA',
    'UGANDA':'AFRICA',
    'ZIMBABWE':'AFRICA',
    'MALI':'AFRICA',
    'SUDAN':'AFRICA',
    'MAURITANIA':'AFRICA', 
    'ETHIOPIA':'AFRICA',
    'RUANDA':'AFRICA',
    'RWANDA':'AFRICA',
    'BURUNDI':'AFRICA', 
    'LYBIA':'AFRICA',
    'MAURITUS IS':'AFRICA',
    'LATVIA':'EU',
    'ESTONIA':'EU',

    'CAMEROON':'AFRICA',
    'IVORY COST':'AFRICA',
    'REUNION':'AFRICA',
    'DOM. REP':'CENTRAL AMERICA',
    'DOM REP.':'CENTRAL AMERICA',
    'DOM REP;':'CENTRAL AMERICA',
    'DOM. REP.':'CENTRAL AMERICA',
    'DOM. REP;':'CENTRAL AMERICA',
    'DOM REP':'CENTRAL AMERICA',
    'TRINIDAD & TOBAGO':'CENTRAL AMERICA',
    'GEORGIA':'ME ASIA',
    'KUWEIT':'ME ASIA',
    'BAHREIN':'ME ASIA',
    'DJBOUTI':'ME ASIA',
    'SAUDI ARABIA ':'ME ASIA',

    
    'BRUNEI':'SE ASIA',
    'MYANMAR':'SE ASIA',
    'MALAYSIA':'SE ASIA',
    'Malaysia':'SE ASIA',
    'PHILIPINES':'SE ASIA',
    'INDIA':'ASIA',
    'BELARUS':'ASIA',

    'NEW ZELAND':'OCEANIA',
    'MAURITUIS':'AFRICA',
    'U.A.E.':'ME ASIA',
    'EAU':'ME ASIA',
    'LEBANNON':'ME ASIA',
    'HOLANDA':'EU',
    'PAKISTAN':'ASIA',
    'PAKISTAN ':'ASIA',
    'RUSSIAN FEDERATION':'ASIA',
    'UNITED STATES':'ASIA',
    'TURKEY ':'ME ASIA',

    'SOUTH KOREA ':'ASIA',
    'JAPAN  ':'ASIA',
    'MALAYSIA  ':'SE ASIA',
    'IRAK':'ME ASIA', 
    'BRASIL':'SOUTH AMERICA',
    'MARRUECOS':'AFRICA',
    'ECUADOR ':'SOUTH AMERICA',
    'PARAGUAY':'SOUTH AMERICA',
    'BRAZIL ':'SOUTH AMERICA',
    'CHILE ':'SOUTH AMERICA',
    'BOLIVIA':'SOUTH AMERICA',
    'MADAGASCAR':'AFRICA',
    'GABON':'AFRICA',
    'SENEGAL ':'AFRICA',
    'GAMBIA':'AFRICA',
    'QATAR':'ME ASIA', 
    'BAHRAIN':'ME ASIA',
    'LEBANON ' :'ME ASIA',
    'BURKINA FASO':'AFRICA',
    'ALGERIA ':'AFRICA',
    'MALAWI':'AFRICA',
    'GUINEA':'AFRICA',
    'TOGO':'AFRICA',
    'LIBERIA':'AFRICA',
    'DJIBOUTI':'AFRICA',
    'VIETNAM ':'SE ASIA',
    "OTH_AFR":'AFRICA',
    "OTH_AMER":"SOUTH AMERICA",
    "OTH_EME":"ME ASIA",
    "OTH_ASIA":"SE ASIA",
    "S. ARABIA":"ME ASIA",
    "WORLD":"WORLD",
    "UNKNOWN":'UNKNOWN',
    "UNITED ARAB EMIRATES + OMAN + SAUDI ARABIA":"ME ASIA"
    }


In [0]:
osa=sp_mgr.read_pd_from_excel('/sites/grp-oilseedssnd/Shared%20Documents/MainFile/Oilseeds_Analytics_version2.xlsm',sheet_name="mtx_sbm")
ARG_SBM = osa.loc[:, (osa.columns[0],) + tuple(osa.columns[1:][osa.iloc[1, 1:].astype(str).str.contains('arg', case=False, na=False)])]
ARG_SBM.columns = ARG_SBM.iloc[0]    # Set first row as header
ARG_SBM = ARG_SBM[1:]                # Drop the first row from the data
ARG_SBM.reset_index(drop=True, inplace=True)  
ARG_SBM = ARG_SBM[2:] 
# Try to convert the first column to datetime
ARG_SBM.iloc[:, 0] = pd.to_datetime(ARG_SBM.iloc[:, 0], errors='coerce')

# Keep only rows where the first column could be parsed as a datetime
ARG_SBM = ARG_SBM[ARG_SBM.iloc[:, 0].notna()]

# Optional: reset index
ARG_SBM.reset_index(drop=True, inplace=True)

ARG_SBM = ARG_SBM.rename(columns={'SBM': 'date'})
ARG_SBM['month']=ARG_SBM['date'].dt.month
ARG_SBM['year']=ARG_SBM['date'].dt.year

current_year = datetime.now().year
# Filter the DataFrame
df_current_year = ARG_SBM[ARG_SBM['year'] == current_year]
# Pivot so that countries are rows and months are columns
pivot_df = df_current_year.set_index('date').drop(columns=['month', 'year']).T
# Set column names to the months
pivot_df.columns = df_current_year['month'].values
pivot_df = pivot_df[~(pivot_df == 0).all(axis=1)]

pivot_df = pivot_df.reset_index().rename(columns={'index': 'Country'})

# Rename first column to 'Country'
df1 = monthly_table.rename(columns={monthly_table.columns[0]: "Country"})
df2 = pivot_df.rename(columns={pivot_df.columns[0]: "Country"})

# Normalize country names to uppercase (or lowercase, your choice)
df1["Country"] = df1["Country"].str.upper()
df2["Country"] = df2["Country"].str.upper()

# Select current month
target_date = datetime.today() - timedelta(days=6)

# Extract the month (as an integer)
month_col = target_date.month


# Extract and rename relevant columns
lineups = df1[["Country", month_col]].rename(columns={month_col: "Lineup"})
forecasts = df2[["Country", month_col]].rename(columns={month_col: "Forecast"})

# Merge
merged = pd.merge(lineups, forecasts, on="Country", how="outer")

# Optional: sort
merged = merged.sort_values("Country").reset_index(drop=True)

merged['Forecast']=merged['Forecast']*1000
merged['Lineup']=merged['Lineup'].round()
merged['Var']=merged['Lineup']-merged['Forecast']

merged['Country'] = merged['Country'].apply(lambda x: country_mapping.get(x, x))
merged['Region'] = merged['Country'].map(country_region_mapping)

# Group by 'Region' and calculate the sum for each region
region_subtotals = merged.groupby('Region').agg({
    'Lineup': 'sum',
    'Forecast': 'sum',
    'Var': 'sum'
}).reset_index()

# Add a column for the subtotal row name
region_subtotals['Country'] = 'Subtotal'

# Append the subtotal rows to the merged dataframe
merged_with_subtotals = pd.concat([merged, region_subtotals], ignore_index=True)

# Sort the dataframe to place subtotal rows at the end of each region
merged_with_subtotals['Region_Order'] = merged_with_subtotals['Region'].map({
    'NORTH AMERICA': 0, 'CENTRAL AMERICA': 1, 'SOUTH AMERICA': 2, 'EU': 3,
    'ME ASIA': 4, 'AFRICA': 5, 'ASIA': 6, 'SE ASIA': 7, 'OCEANIA': 8,'UNKNOWN':9,"WORLD":10
})

merged_with_subtotals = merged_with_subtotals.sort_values(by=['Region_Order', 'Country'])

# Drop the 'Region_Order' column
merged_with_subtotals = merged_with_subtotals.drop(columns=['Region_Order'])


# Map the regions to the 'Country' column
merged['Region'] = merged['Country'].map(country_region_mapping)

# Add subtotals by region
subtotal_by_region = merged.groupby('Region').sum().reset_index()
subtotal_by_region['Country'] = 'Subtotal'

# Append the subtotal row to the original DataFrame
merged = pd.concat([merged, subtotal_by_region], ignore_index=True)

region_order = [
    'NORTH AMERICA', 'CENTRAL AMERICA', 'SOUTH AMERICA',
    'EU', 'ME ASIA', 'AFRICA', 'ASIA', 'SE ASIA', 'OCEANIA','UNKNOWN',"WORLD"
]

# Ensure Region is a categorical column with order
merged_with_subtotals['Region'] = pd.Categorical(
    merged_with_subtotals['Region'],
    categories=region_order,
    ordered=True
)

# Replace 'Subtotal' country entries with 'Subtotal [Region]'
merged_with_subtotals.loc[
    merged_with_subtotals['Country'] == 'Subtotal',
    'Country'
] = 'Subtotal ' + merged_with_subtotals['Region'].astype(str)

# Sort values by Region and then within Region put Subtotal at the end
def custom_sort(df):
    # Put all non-subtotals first, then the subtotal
    subtotals = df[df['Country'].str.startswith('Subtotal')]
    others = df[~df['Country'].str.startswith('Subtotal')]
    return pd.concat([others, subtotals])

# Apply the sorting logic by region
merged_with_subtotals = (
    merged_with_subtotals
    .sort_values(['Region', 'Country'])  # Preliminary sort
    .groupby('Region', group_keys=False)
    .apply(custom_sort)
)
merged_with_subtotals = merged_with_subtotals[merged_with_subtotals['Country'] != 'Subtotal UNKNOWN']
merged_with_subtotals = merged_with_subtotals[merged_with_subtotals['Country'] != 'Subtotal WORLD']

# Optional: reset index or keep Region as index
merged_with_subtotals.set_index(['Region', 'Country'], inplace=True)

total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

# Set the value for the WORLD row
merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] = total_lineup


merged_with_subtotals = merged_with_subtotals.rename(columns={'Forecast': 'BS'})

# Now set the 'Var' column for the 'WORLD' row
merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Var'] = merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] - merged_with_subtotals.loc[('WORLD', 'WORLD'), 'BS']


merged_with_subtotals_ht=merged_with_subtotals.fillna(0)

# Drop rows where both Lineup and Forecast are zero
merged_with_subtotals_ht= merged_with_subtotals_ht[~((merged_with_subtotals_ht['Lineup'] == 0) & (merged_with_subtotals_ht['BS'] == 0))]

# Format numeric values with thousand separators and no decimals
merged_with_subtotals_ht[['Lineup', 'BS', 'Var']] = merged_with_subtotals_ht[['Lineup', 'BS', 'Var']].applymap(lambda x: f"{x:,.0f}")

total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

html_byregions=merged_with_subtotals_ht.to_html()

### CHART OVERVIEW

In [0]:
BS_world = merged_with_subtotals.reset_index()
BS_value = BS_world.loc[BS_world['Country'] == 'WORLD', 'BS'].values[0]

filtered_df_SBM = filtered_df_SBM[filtered_df_SBM['Origin']=='ARG']

sailed_df = filtered_df_SBM[filtered_df_SBM['Status'] == 'SAILED'].copy()

# Step 2: Make sure 'Date' is a datetime column
sailed_df['Date'] = pd.to_datetime(sailed_df['Date'])

# Step 3: Sort by 'Date'
sailed_df = sailed_df.sort_values(by='Date')

# Step 4: Compute the cumulative sum of 'Quantity'
sailed_df['Sailed'] = sailed_df['Quantity'].cumsum()

# Step 1: Filter for Status being 'SAILED' or 'ANNOUNCED'
forecast_df = filtered_df_SBM[filtered_df_SBM['Status'].isin(['SAILED', 'ANNOUNCED'])].copy()

# Step 2: Make sure 'Date' is datetime
forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])

# Step 3: Sort by date
forecast_df = forecast_df.sort_values(by='Date')

# Step 4: Compute cumulative sum of Quantity
forecast_df['Forecasts'] = forecast_df['Quantity'].cumsum()

daily_totals_sailed = sailed_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_sailed['Sailed'] = daily_totals_sailed['Quantity'].cumsum()

daily_totals_forecast = forecast_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()

num_days = len(daily_totals_forecast)

# Create the constant daily value from BS
daily_totals_forecast['BS_per_day'] = BS_value / num_days

# Compute the cumulative sum
daily_totals_forecast['BS'] = daily_totals_forecast['BS_per_day'].cumsum()

# Get the last date's value to annotate
last_sailed = daily_totals_sailed.iloc[-1]
last_forecast = daily_totals_forecast.iloc[-1]


In [0]:

overview = go.Figure()

# Sailed line
overview.add_trace(go.Scatter(
    x=daily_totals_sailed['Date'],
    y=daily_totals_sailed['Sailed'],
    mode='lines+markers',
    name='Sailed',
    line=dict(color='blue')
))

# Forecast line
overview.add_trace(go.Scatter(
    x=daily_totals_forecast['Date'],
    y=daily_totals_forecast['Forecasts'],
    mode='lines+markers',
    name='Lineup (sailed+announced)',
    line=dict(color='orange', dash='dash')
))

# Forecast line
overview.add_trace(go.Scatter(
    x=daily_totals_forecast['Date'],
    y=daily_totals_forecast['BS'],
    mode='lines+markers',
    name='BS numbers',
    line=dict(color='red', dash='dashdot')
))


# Add annotations
overview.add_annotation(
    x=last_sailed['Date'],
    y=last_sailed['Sailed'],
    text=f"{int(last_sailed['Sailed']/1000):,} kmt",
    showarrow=True,
    arrowhead=1,
    ax=0,
    ay=-60,
    font=dict(color="blue")
)

overview.add_annotation(
    x=last_forecast['Date'],
    y=last_forecast['Forecasts'],
    text=f"{int(last_forecast['Forecasts']/1000):,} kmt",
    showarrow=True,
    arrowhead=1,
    ax=0,
    ay=-40,
    font=dict(color="orange")
)

overview.add_annotation(
    x=last_forecast['Date'],
    y=last_forecast['BS'],
    text=f"{int(last_forecast['BS']/1000):,} kmt",
    showarrow=True,
    arrowhead=1,
    ax=70,
    ay=-20,
    font=dict(color="red")
)
# Layout
overview.update_layout(
    title='Sailed vs Forecasts',
    xaxis_title='Date',
    yaxis_title='Quantity',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)

# Export to image
pio.write_image(overview, "sailed_vs_forecast.png", width=1000, height=600)

overview.show()

In [0]:


anncd = monthly_table_ancd[["Destination", month_col]].rename(columns={month_col: "Announced"})
sailed = monthly_table_sailed[["Destination", month_col]].rename(columns={month_col: "Sailed"})

anncd.rename(columns={'Destination': 'Country'}, inplace=True)
sailed.rename(columns={'Destination': 'Country'}, inplace=True)

anncd['Country'] = anncd['Country'].apply(lambda x: country_mapping.get(x, x))
sailed['Country'] = sailed['Country'].apply(lambda x: country_mapping.get(x, x))

merged_bar=pd.merge(anncd, sailed, on='Country')
merged_bar=pd.merge(merged, merged_bar, on='Country')
merged_bar=merged_bar.sort_values('Region')

# Filter out rows where Sailed, Announced, and Forecast are all 0 or NaN
merged_bar = merged_bar[~(((merged_bar['Sailed'].fillna(0) == 0) & (merged_bar['Announced'].fillna(0) == 0) & (merged_bar['Forecast'].fillna(0) == 0)))]

merged_bar['Total'] = merged_bar['Sailed'].fillna(0) + merged_bar['Announced'].fillna(0)

# Sort by this total and keep only top 20
merged_bar = merged_bar.sort_values('Total', ascending=False).head(20)

# Optional: sort again for cleaner plotting (descending)
merged_bar = merged_bar.sort_values('Total', ascending=True) 

# Step 1: Create the base chart with Sailed + Announced side by side
bar = go.Figure()

# Sailed
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Sailed'],
    orientation='h',
    name='Sailed',
    marker=dict(color='steelblue'),
    offsetgroup=0,
    base=0,
    text=merged_bar['Sailed'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=30)  
))

# Announced
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Announced'],
    orientation='h',
    name='Announced',
    marker=dict(color='orange'),
    offsetgroup=0,
    base=merged_bar['Sailed'],
    text=merged_bar['Announced'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=30)  
))

# Step 2: Add Forecast with overlay
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Forecast'],
    orientation='h',
    name='BS',
    marker=dict(color='rgba(255, 0, 0, 0.4)'),  # Transparent red
    offsetgroup=1,
    base=0,
    opacity=0.4,
    text=merged_bar['Forecast'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=60)  
))

# Layout
bar.update_layout(
    barmode='group',  # This allows Forecast to overlap
    title='Sailed et Announced vs Balance Sheet by Country',
    xaxis_title='Quantity (mt)',
    yaxis_title='Country',
    template='plotly_white',
    height=900
)

bar.show()


### TOP 10

In [0]:
# Add new column for the total of other columns
monthly_table['Total'] = monthly_table.iloc[:, 1:].sum(axis=1)

# Step 2: Select top 10 countries with the highest total
top_10_countries = monthly_table.nlargest(10, 'Total')
# Step 3: Map column numbers to month names (1 = January, 2 = February, ..., up to current month)
current_month = datetime.now().month  # Get current month number (1-12)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_labels = month_names[:current_month]  # Only up to the current month

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# Get the number of month columns (excluding the first column 'Destination' and the last column 'Total')
num_months = top_10_countries.shape[1] - 2  # exclude 'Destination' and 'Total'

# Rename columns accordingly
new_columns = ['Destination'] + month_names[:num_months] + ['Total']
top_10_countries.columns = new_columns

# Display the result
top_10_countries.head()


In [0]:

# Step 4: Create the plot
top = go.Figure()

# Add a trace for each of the top 10 countries
for country in top_10_countries['Destination']:
    country_data = top_10_countries[top_10_countries['Destination'] == country]
    top.add_trace(go.Scatter(
        x=month_labels,
        y=country_data.iloc[0, 1:current_month+1],
        mode='lines+markers',
        name=country
    ))

# Update the layout
top.update_layout(
    title='',
    xaxis_title='Month',
    yaxis_title='Quantity (mt)',
    template='plotly_white'
)

top_10_countries = top_10_countries.applymap(lambda x: f"{x:,.0f}" if isinstance(x, (int, float)) else x)
top_10_countries = top_10_countries.reset_index(drop=True)
top_10_countries=top_10_countries[['Destination', 'Total']]
top_10_countries = top_10_countries.to_html()

### COORDINATOR SHIPPER

In [0]:
# Get today's date
today = datetime.today()

# Get the first day of the current month
first_day_of_current_month = today.replace(day=1)

# Calculate the date for the current month - 6 days
current_month_minus_6_days = first_day_of_current_month - timedelta(days=6)

# Filter the DataFrame for dates within the last 6 days of the current month
filtered_df = filtered_df_SBM_year[filtered_df_SBM_year['Date'] >= current_month_minus_6_days]

# Group by 'Shipper' and sum 'Quantity'
grouped_by_shipper = filtered_df.groupby('Shipper')['Quantity'].sum().reset_index()

# Group by 'Coordinator' and sum 'Quantity'
grouped_by_coordinator = filtered_df.groupby('Coordinator')['Quantity'].sum().reset_index()

# Calculate the total sum of Quantity for all coordinators
total_sum_coordinator = grouped_by_coordinator['Quantity'].sum()

# Calculate the percentage for each coordinator
grouped_by_coordinator['Percentage'] = (grouped_by_coordinator['Quantity'] / total_sum_coordinator) * 100

total_sum_shipper = grouped_by_shipper['Quantity'].sum()

# Calculate the percentage for each shipper
grouped_by_shipper['Percentage'] = (grouped_by_shipper['Quantity'] / total_sum_shipper) * 100


top_10_coordinators = grouped_by_coordinator.sort_values('Percentage', ascending=False).head(10)
top_10_shippers = grouped_by_shipper.sort_values('Percentage', ascending=False).head(10)

# Merge the top 10 Coordinators and Shippers
merged_df = pd.merge(top_10_coordinators[['Coordinator', 'Percentage']], 
                     top_10_shippers[['Shipper', 'Percentage']], 
                     how='outer', 
                     left_on='Coordinator', 
                     right_on='Shipper', 
                     suffixes=('_coordinator', '_shipper'))

# Create a Plotly figure
cie = go.Figure()

# Add a bar chart for Coordinators
cie.add_trace(go.Bar(
    x=merged_df['Coordinator'].fillna(merged_df['Shipper']),
    y=merged_df['Percentage_coordinator'],
    name='Coordinator',  # This adds the name to the legend
    marker_color='blue',  # Color for Coordinators
    opacity=0.6,
    text=merged_df['Percentage_coordinator'].round(2),  # Show percentage values
    textposition='outside',  # Position the text outside the bars
))

# Add a bar chart for Shippers
cie.add_trace(go.Bar(
    x=merged_df['Coordinator'].fillna(merged_df['Shipper']),
    y=merged_df['Percentage_shipper'],
    name='Shipper',  # This adds the name to the legend
    marker_color='orange',  # Color for Shippers
    opacity=0.6,
    text=merged_df['Percentage_shipper'].round(2),  # Show percentage values
    textposition='outside',  # Position the text outside the bars
))

# Set layout details
cie.update_layout(
    title="Top 10 Coordinators and Shippers",
    barmode='group',
    xaxis_title="",
    yaxis_title="%",
    showlegend=True,  # Ensure legend is displayed
    template="plotly_white"
)



### SEPARATED

In [0]:

# Get top 10 Coordinators and Shippers
top_10_coordinators = grouped_by_coordinator.sort_values('Percentage', ascending=False).head(10)
top_10_shippers = grouped_by_shipper.sort_values('Percentage', ascending=False).head(10)

# Chart 1: Top 10 Coordinators
fig_coordinators = go.Figure()

fig_coordinators.add_trace(go.Bar(
    x=top_10_coordinators['Coordinator'],
    y=top_10_coordinators['Percentage'],
    name='Coordinator',
    marker_color='blue',
    text=top_10_coordinators['Percentage'].round(2),
    textposition='outside',
))

fig_coordinators.update_layout(
    title="Top 10 Coordinators",
    xaxis_title="Coordinator",
    yaxis_title="%",
    showlegend=False,
    template="plotly_white"
)

# Chart 2: Top 10 Shippers
fig_shippers = go.Figure()

fig_shippers.add_trace(go.Bar(
    x=top_10_shippers['Shipper'],
    y=top_10_shippers['Percentage'],
    name='Shipper',
    marker_color='orange',
    text=top_10_shippers['Percentage'].round(2),
    textposition='outside',
))

fig_shippers.update_layout(
    title="Top 10 Shippers",
    xaxis_title="Shipper",
    yaxis_title="%",
    showlegend=False,
    template="plotly_white"
)

# Show the plots
fig_coordinators.show()
fig_shippers.show()


In [0]:
# Chart 1: Top 10 Coordinators (Pie Chart)
fig_coordinators = go.Figure()

fig_coordinators.add_trace(go.Pie(
    labels=top_10_coordinators['Coordinator'],
    values=top_10_coordinators['Percentage'],
    textinfo='label+percent',
    insidetextorientation='radial',
    marker=dict(colors=px.colors.sequential.Blues)
))

fig_coordinators.update_layout(
    title="Top 10 Coordinators (Pie Chart)",
    template="plotly_white"
)

# Chart 2: Top 10 Shippers (Pie Chart)
fig_shippers = go.Figure()

fig_shippers.add_trace(go.Pie(
    labels=top_10_shippers['Shipper'],
    values=top_10_shippers['Percentage'],
    textinfo='label+percent',
    insidetextorientation='radial',
    marker=dict(colors=px.colors.sequential.Blues)
))

fig_shippers.update_layout(
    title="Top 10 Shippers (Pie Chart)",
    template="plotly_white"
)

# Show the plots
fig_coordinators.show()
fig_shippers.show()


### NEXT MONTH

In [0]:
ref_date = datetime.today() - timedelta(days=5)

# Step 2: Extract year and month of that reference date
ref_year = ref_date.year
ref_month = ref_date.month

# Step 1: Determine next month and year
if ref_month == 12:
    next_month = 1
    next_year = ref_year + 1
else:
    next_month = ref_month + 1
    next_year = ref_year

# Step 2: Filter for ref_month of ref_year OR next_month of next_year
filtered_df = history[((history['Date'].dt.year == next_year) & (history['Date'].dt.month == next_month))].copy()


In [0]:
filtered_df_SBM=filtered_df[filtered_df['Product']=='SBM']
filtered_df_SBM['Date'] = filtered_df_SBM['Date'].dt.date

In [0]:


# Group announced quantities by region
announced = filtered_df_SBM[filtered_df_SBM['Status'] == 'ANNOUNCED'].groupby('Region', as_index=False)['Quantity'].sum()
announced = announced.rename(columns={'Quantity': 'Quantity_Announced'})

# Merge both
region_totals = announced.copy()

# Add total column
region_totals['Total'] = region_totals['Quantity_Announced']

# Optional: sort by total
region_totals = region_totals.sort_values(by='Total', ascending=False)

filtered_df_year = history[(history['Date'].dt.year == next_year)]
filtered_df_SBM_year=filtered_df_year[filtered_df_year['Product']=='SBM']
filtered_df_SBM_year=filtered_df_SBM_year[filtered_df_SBM_year['Origin']=='ARG']


# Ensure Date is datetime type if not already
filtered_df_SBM_year['Date'] = pd.to_datetime(filtered_df_SBM_year['Date'])

# Add Month column (numerical)
filtered_df_SBM_year['Month'] = filtered_df_SBM_year['Date'].dt.month

# Clean country names if needed
filtered_df_SBM_year['Destination'] = filtered_df_SBM_year['Destination'].str.upper().str.strip()

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table = (
    filtered_df_SBM_year
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
    .sort_index(axis=1)     # Ensure months go 1 to 12
)

# Optional: round values
monthly_table = monthly_table.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table.columns.name = None
monthly_table = monthly_table.reset_index()


# Clean country names if needed
filtered_df_SBM_year['Destination'] = filtered_df_SBM_year['Destination'].str.upper().str.strip()

filtered_df_SBM_year_anc=filtered_df_SBM_year[filtered_df_SBM_year['Status']=='ANNOUNCED']



# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table_ancd = (
    filtered_df_SBM_year_anc
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
    .sort_index(axis=1)     # Ensure months go 1 to 12
)

# Optional: round values
monthly_table_ancd = monthly_table_ancd.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table_ancd.columns.name = None
monthly_table_ancd = monthly_table_ancd.reset_index()

from datetime import datetime

# Step 1: Add 'Month' column if missing
filtered_df_SBM_year['Month'] = filtered_df_SBM_year['Date'].dt.month

# Step 2: Get current month + 1
cutoff_month = (datetime.today().month + 1)

# Step 3: Filter up to current month + 1
filtered_df_SBM_cut = filtered_df_SBM_year[filtered_df_SBM_year['Month'] <= cutoff_month]

# Step 4: Create pivot table by Region and Month
quantity_by_region_month = pd.pivot_table(
    filtered_df_SBM_cut,
    values='Quantity',
    index='Region',
    columns='Month',
    aggfunc='sum',
    fill_value=0
)

# Step 5: Replace month numbers with names (Jan, Feb, etc.)
quantity_by_region_month.columns = [datetime(1900, m, 1).strftime('%b') for m in quantity_by_region_month.columns]





In [0]:
# Merge
merged = pd.merge(lineups, forecasts, on="Country", how="outer")
merged

In [0]:
from datetime import datetime

# Get today's date
today = datetime.today()

if today.day > 20:
    month_col = next_month


    # Extract and rename relevant columns
    lineups = df1[["Country", month_col]].rename(columns={month_col: "Lineup"})
    forecasts = df2[["Country", month_col]].rename(columns={month_col: "Forecast"})

    # Merge
    merged = pd.merge(lineups, forecasts, on="Country", how="outer")

    # Optional: sort
    merged = merged.sort_values("Country").reset_index(drop=True)

    merged['Forecast']=merged['Forecast']*1000
    merged['Lineup']=merged['Lineup'].round()
    merged['Var']=merged['Lineup']-merged['Forecast']

    merged['Country'] = merged['Country'].apply(lambda x: country_mapping.get(x, x))
    merged['Region'] = merged['Country'].map(country_region_mapping)

    # Group by 'Region' and calculate the sum for each region
    region_subtotals = merged.groupby('Region').agg({
        'Lineup': 'sum',
        'Forecast': 'sum',
        'Var': 'sum'
    }).reset_index()

    # Add a column for the subtotal row name
    region_subtotals['Country'] = 'Subtotal'

    # Append the subtotal rows to the merged dataframe
    merged_with_subtotals = pd.concat([merged, region_subtotals], ignore_index=True)

    # Sort the dataframe to place subtotal rows at the end of each region
    merged_with_subtotals['Region_Order'] = merged_with_subtotals['Region'].map({
        'NORTH AMERICA': 0, 'CENTRAL AMERICA': 1, 'SOUTH AMERICA': 2, 'EU': 3,
        'ME ASIA': 4, 'AFRICA': 5, 'ASIA': 6, 'SE ASIA': 7, 'OCEANIA': 8,"WORLD":9
    })

    merged_with_subtotals = merged_with_subtotals.sort_values(by=['Region_Order', 'Country'])

    # Drop the 'Region_Order' column
    merged_with_subtotals = merged_with_subtotals.drop(columns=['Region_Order'])


    # Map the regions to the 'Country' column
    merged['Region'] = merged['Country'].map(country_region_mapping)

    # Add subtotals by region
    subtotal_by_region = merged.groupby('Region').sum().reset_index()
    subtotal_by_region['Country'] = 'Subtotal'

    # Append the subtotal row to the original DataFrame
    merged = pd.concat([merged, subtotal_by_region], ignore_index=True)

    region_order = [
        'NORTH AMERICA', 'CENTRAL AMERICA', 'SOUTH AMERICA',
        'EU', 'ME ASIA', 'AFRICA', 'ASIA', 'SE ASIA', 'OCEANIA',"WORLD"
    ]

    # Ensure Region is a categorical column with order
    merged_with_subtotals['Region'] = pd.Categorical(
        merged_with_subtotals['Region'],
        categories=region_order,
        ordered=True
    )

    # Replace 'Subtotal' country entries with 'Subtotal [Region]'
    merged_with_subtotals.loc[
        merged_with_subtotals['Country'] == 'Subtotal',
        'Country'
    ] = 'Subtotal ' + merged_with_subtotals['Region'].astype(str)

    # Sort values by Region and then within Region put Subtotal at the end
    def custom_sort(df):
        # Put all non-subtotals first, then the subtotal
        subtotals = df[df['Country'].str.startswith('Subtotal')]
        others = df[~df['Country'].str.startswith('Subtotal')]
        return pd.concat([others, subtotals])

    # Apply the sorting logic by region
    merged_with_subtotals = (
        merged_with_subtotals
        .sort_values(['Region', 'Country'])  # Preliminary sort
        .groupby('Region', group_keys=False)
        .apply(custom_sort)
    )

    merged_with_subtotals = merged_with_subtotals[merged_with_subtotals['Country'] != 'Subtotal WORLD']

    # Optional: reset index or keep Region as index
    merged_with_subtotals.set_index(['Region', 'Country'], inplace=True)

    total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

    # Set the value for the WORLD row
    merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] = total_lineup


    merged_with_subtotals = merged_with_subtotals.rename(columns={'Forecast': 'BS'})

    # Now set the 'Var' column for the 'WORLD' row
    merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Var'] = merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] - merged_with_subtotals.loc[('WORLD', 'WORLD'), 'BS']


    merged_with_subtotals_ht=merged_with_subtotals.fillna(0)

    # Drop rows where both Lineup and Forecast are zero
    merged_with_subtotals_ht= merged_with_subtotals_ht[~((merged_with_subtotals_ht['Lineup'] == 0) & (merged_with_subtotals_ht['BS'] == 0))]

    # Format numeric values with thousand separators and no decimals
    merged_with_subtotals_ht[['Lineup', 'BS', 'Var']] = merged_with_subtotals_ht[['Lineup', 'BS', 'Var']].applymap(lambda x: f"{x:,.0f}")

    total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

    html_byregions_next_month=merged_with_subtotals_ht.to_html()

    BS_world = merged_with_subtotals.reset_index()
    BS_value = BS_world.loc[BS_world['Country'] == 'WORLD', 'BS'].values[0]

    filtered_df_SBM = filtered_df_SBM[filtered_df_SBM['Origin']=='ARG']


    # Step 1: Filter for Status being 'SAILED' or 'ANNOUNCED'
    forecast_df = filtered_df_SBM[filtered_df_SBM['Status'].isin(['SAILED', 'ANNOUNCED'])].copy()

    # Step 2: Make sure 'Date' is datetime
    forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])

    # Step 3: Sort by date
    forecast_df = forecast_df.sort_values(by='Date')

    # Step 4: Compute cumulative sum of Quantity
    forecast_df['Forecasts'] = forecast_df['Quantity'].cumsum()


    daily_totals_forecast = forecast_df.groupby('Date', as_index=False)['Quantity'].sum()

    # Step 2: Compute the cumulative sum
    daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()

    num_days = len(daily_totals_forecast)

    # Create the constant daily value from BS
    daily_totals_forecast['BS_per_day'] = BS_value / num_days

    # Compute the cumulative sum
    daily_totals_forecast['BS'] = daily_totals_forecast['BS_per_day'].cumsum()

    last_forecast = daily_totals_forecast.iloc[-1]

    overview_next_month = go.Figure()

    # Forecast line
    overview_next_month.add_trace(go.Scatter(
        x=daily_totals_forecast['Date'],
        y=daily_totals_forecast['Forecasts'],
        mode='lines+markers',
        name='Lineup (sailed+announced)',
        line=dict(color='orange', dash='dash')
    ))

    # Forecast line
    overview_next_month.add_trace(go.Scatter(
        x=daily_totals_forecast['Date'],
        y=daily_totals_forecast['BS'],
        mode='lines+markers',
        name='BS numbers',
        line=dict(color='red', dash='dashdot')
    ))

    overview_next_month.add_annotation(
        x=last_forecast['Date'],
        y=last_forecast['Forecasts'],
        text=f"{int(last_forecast['Forecasts']/1000):,} kmt",
        showarrow=True,
        arrowhead=1,
        ax=0,
        ay=-40,
        font=dict(color="orange")
    )

    overview_next_month.add_annotation(
        x=last_forecast['Date'],
        y=last_forecast['BS'],
        text=f"{int(last_forecast['BS']/1000):,} kmt",
        showarrow=True,
        arrowhead=1,
        ax=70,
        ay=-20,
        font=dict(color="red")
    )
    # Layout
    overview_next_month.update_layout(
        title='Announced and BS Forecasts for next month',
        xaxis_title='Date',
        yaxis_title='Quantity',
        template='plotly_white',
        legend=dict(x=0.01, y=0.99)
    )

    # Export to image
    pio.write_image(overview_next_month, "sailed_vs_forecast.png", width=1000, height=600)

        

    anncd = monthly_table_ancd[["Destination", month_col]].rename(columns={month_col: "Announced"})

    anncd.rename(columns={'Destination': 'Country'}, inplace=True)


    anncd['Country'] = anncd['Country'].apply(lambda x: country_mapping.get(x, x))



    merged_bar=pd.merge(merged, anncd, on='Country')
    merged_bar=merged_bar.sort_values('Region')

    merged_bar = merged_bar[~(((merged_bar['Announced'].fillna(0) == 0) & 
                            (merged_bar['Forecast'].fillna(0) == 0)))]



    # Step 1: Create the base chart with Sailed + Announced side by side
    bar_next_month = go.Figure()


    # Announced
    bar_next_month.add_trace(go.Bar(
        y=merged_bar['Country'],
        x=merged_bar['Announced'],
        orientation='h',
        name='Announced',
        marker=dict(color='orange'),
        offsetgroup=0,
        text=merged_bar['Announced'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
        textposition='inside',  # Position text inside the bar
        textfont=dict(size=30)  
    ))

    # Step 2: Add Forecast with overlay
    bar_next_month.add_trace(go.Bar(
        y=merged_bar['Country'],
        x=merged_bar['Forecast'],
        orientation='h',
        name='BS',
        marker=dict(color='rgba(255, 0, 0, 0.4)'),  # Transparent red
        offsetgroup=1,
        base=0,
        opacity=0.4,
        text=merged_bar['Forecast'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
        textposition='inside',  # Position text inside the bar
        textfont=dict(size=60)  
    ))

    # Layout
    bar_next_month.update_layout(
        barmode='group',  # This allows Forecast to overlap
        title='Announced vs Balance Sheet by Country',
        xaxis_title='Quantity (mt)',
        yaxis_title='Country',
        template='plotly_white',
        height=900
    )

    bar_next_month.show()



    ### PLAYERS

    ref_date = datetime.today() - timedelta(days=5)

    # Step 2: Extract year and month of that reference date
    ref_year = ref_date.year
    ref_month = ref_date.month

    # Step 1: Determine next month and year
    if ref_month == 12:
        next_month = 1
        next_year = ref_year + 1
    else:
        next_month = ref_month + 1
        next_year = ref_year

    # Step 1: Build the first day of next month
    first_day_of_next_month = datetime(year=next_year, month=next_month, day=1)

    # Step 2: Filter the DataFrame for dates on or after the first day of next month
    filtered_df = filtered_df_SBM_year[filtered_df_SBM_year['Date'] >= first_day_of_next_month]

    # Group by 'Shipper' and sum 'Quantity'
    grouped_by_shipper = filtered_df.groupby('Shipper')['Quantity'].sum().reset_index()

    # Group by 'Coordinator' and sum 'Quantity'
    grouped_by_coordinator = filtered_df.groupby('Coordinator')['Quantity'].sum().reset_index()

    # Calculate the total sum of Quantity for all coordinators
    total_sum_coordinator = grouped_by_coordinator['Quantity'].sum()

    # Calculate the percentage for each coordinator
    grouped_by_coordinator['Percentage'] = (grouped_by_coordinator['Quantity'] / total_sum_coordinator) * 100

    total_sum_shipper = grouped_by_shipper['Quantity'].sum()

    # Calculate the percentage for each shipper
    grouped_by_shipper['Percentage'] = (grouped_by_shipper['Quantity'] / total_sum_shipper) * 100


    top_10_coordinators = grouped_by_coordinator.sort_values('Percentage', ascending=False).head(10)
    top_10_shippers = grouped_by_shipper.sort_values('Percentage', ascending=False).head(10)

    # Merge the top 10 Coordinators and Shippers
    merged_df = pd.merge(top_10_coordinators[['Coordinator', 'Percentage']], 
                        top_10_shippers[['Shipper', 'Percentage']], 
                        how='outer', 
                        left_on='Coordinator', 
                        right_on='Shipper', 
                        suffixes=('_coordinator', '_shipper'))

    # Create a Plotly figure
    cie = go.Figure()

    # Add a bar chart for Coordinators
    cie.add_trace(go.Bar(
        x=merged_df['Coordinator'].fillna(merged_df['Shipper']),
        y=merged_df['Percentage_coordinator'],
        name='Coordinator',  # This adds the name to the legend
        marker_color='blue',  # Color for Coordinators
        opacity=0.6,
        text=merged_df['Percentage_coordinator'].round(2),  # Show percentage values
        textposition='outside',  # Position the text outside the bars
    ))

    # Add a bar chart for Shippers
    cie.add_trace(go.Bar(
        x=merged_df['Coordinator'].fillna(merged_df['Shipper']),
        y=merged_df['Percentage_shipper'],
        name='Shipper',  # This adds the name to the legend
        marker_color='orange',  # Color for Shippers
        opacity=0.6,
        text=merged_df['Percentage_shipper'].round(2),  # Show percentage values
        textposition='outside',  # Position the text outside the bars
    ))

    # Set layout details
    cie.update_layout(
        title="Top 10 Coordinators and Shippers",
        barmode='group',
        xaxis_title="",
        yaxis_title="%",
        showlegend=True,  # Ensure legend is displayed
        template="plotly_white"
    )

    
    # Get top 10 Coordinators and Shippers
    top_10_coordinators = grouped_by_coordinator.sort_values('Percentage', ascending=False).head(10)
    top_10_shippers = grouped_by_shipper.sort_values('Percentage', ascending=False).head(10)

    # Chart 1: Top 10 Coordinators
    fig_coordinators_next_month = go.Figure()

    fig_coordinators_next_month.add_trace(go.Bar(
        x=top_10_coordinators['Coordinator'],
        y=top_10_coordinators['Percentage'],
        name='Coordinator',
        marker_color='blue',
        text=top_10_coordinators['Percentage'].round(2),
        textposition='outside',
    ))

    fig_coordinators_next_month.update_layout(
        title="Top 10 Coordinators",
        xaxis_title="Coordinator",
        yaxis_title="%",
        showlegend=False,
        template="plotly_white"
    )

    # Create HTML email content with a properly structured table
    from datetime import datetime, timedelta

    # Today's date minus 5 days
    target_date = datetime.today() - timedelta(days=5)

    # Month name (e.g., "April")
    current_month = target_date.strftime("%B")

    ref_date = datetime.today() - timedelta(days=5)

    # Step 2: Extract year and month of that reference date
    ref_year = ref_date.year
    ref_month = ref_date.month

    # Step 1: Determine next month and year
    if ref_month == 12:
        next_month = 1
        next_year = ref_year + 1
    else:
        next_month = ref_month + 1
        next_year = ref_year


    first_day_of_next_month = datetime(year=next_year, month=next_month, day=1)
    next_month = first_day_of_next_month.strftime("%B")

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>SBM Lineups Overview </title>
        <style>
            .container {{
                justify-content: space-between;
                align-items: flex-start;
                width: 100%;
                gap: 10px;
            }}
            .chart-container, .table-container, .extra-container {{
                flex: 0;
                padding: 10px;
                height: 100%;
            }}
            .table-container {{
                text-align: center;
                margin: auto;
                flex: 1; 
                min-width: 300px; 
            }}
            table {{
                width: 60%;
                border-collapse: collapse;
            }}
            th, td {{
                border: 1px solid black;
                padding: 8px;
                text-align: center;
            }}
            th {{
                background-color: #f2f2f2;
            }}
            img {{
                width: 100%;
                height: auto;
                min-width: 750px; /* Set a minimum width */
                min-height: 750px; 
                object-fit: contain;
            }}
            .wide-table {{
                width: 100%;
                table-layout: fixed;
            }}

            .wide-table th, .wide-table td {{
                padding: 8px;
                width: 40%;
                min-width: 100px;
                white-space: nowrap;
            }}
            
        </style>
    </head>
    <body>
        <h1>{current_month} Exports</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Exports Overview</h2>
                    <img src="chart1" alt="SBM Exports overview">
                </td>
            </tr>
        </table>
        <h1>{current_month} Exports by Destinations</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Destinations</h2>
                    <img src="chart2" alt="Board Crush Chart">
                </td>
                <!-- Table Section -->
                <td class="table-container">
                    {html_byregions}  <!-- Insert DataFrame as an HTML table -->
                </td>
            </tr>
        </table>
        <h1>Top 10 Destinations</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Main Destinations</h2>
                    <img src="chart3" alt="top 10">
                </td>
                <!-- Table Section -->
                <td class="table-container">
                    <h2>Accumulated 2025 exports of main destinations</h2>
                    {top_10_countries}  <!-- Insert DataFrame as an HTML table -->
                </td>
            </tr>
        </table>
        <h1>Top 10 players</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <img src="chart4" alt="Top companies">
                </td>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <img src="chart5" alt="Top companies">
                </td>
            </tr>
        </table>

        <h1>{next_month} Exports</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Exports Overview</h2>
                    <img src="chart6" alt="SBM Exports overview">
                </td>
            </tr>
        </table>
        <h1>{next_month} Exports by Destinations</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Destinations</h2>
                    <img src="chart7" alt="Board Crush Chart">
                </td>
                <!-- Table Section -->
                <td class="table-container">
                    {html_byregions_next_month}  <!-- Insert DataFrame as an HTML table -->
                </td>
            </tr>
        </table>
        <h1>Top 10 players</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <img src="chart9" alt="Top companies">
                </td>
            </tr>
        </table>
    </body>
    </html>
    """

    adress_test=['florian.girardi-ext@ldc.com','michelle.lambrechts@ldc.com',"SANTIAGO.CARBAJAL@ldc.com"]

    moi=['florian.girardi-ext@ldc.com']

    # Send email with embedded chart and table
    LDCDataAccessLayerPy.mail.mail_send(
        to=adress_test,
        subject=f'SBM Lineups {datetime.now().strftime("%d-%m")}',
        from_addr="florian.girardi-ext@ldc.com",
        body=html_content,
        mime_type="html",
        html_images={"chart1": overview,"chart2":bar,'chart3':top,'chart4':fig_shippers,'chart5':fig_coordinators,'chart6':overview_next_month,'chart7':bar_next_month,'chart9':fig_coordinators_next_month}  # Embedding the chart image
    )

else:
        # Create HTML email content with a properly structured table
    from datetime import datetime, timedelta

    # Today's date minus 5 days
    target_date = datetime.today() - timedelta(days=5)

    # Month name (e.g., "April")
    current_month = target_date.strftime("%B")

    ref_date = datetime.today() - timedelta(days=5)

    # Step 2: Extract year and month of that reference date
    ref_year = ref_date.year
    ref_month = ref_date.month

    # Step 1: Determine next month and year
    if ref_month == 12:
        next_month = 1
        next_year = ref_year + 1
    else:
        next_month = ref_month + 1
        next_year = ref_year


    first_day_of_next_month = datetime(year=next_year, month=next_month, day=1)
    next_month = first_day_of_next_month.strftime("%B")

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>SBM Lineups Overview </title>
        <style>
            .container {{
                justify-content: space-between;
                align-items: flex-start;
                width: 100%;
                gap: 10px;
            }}
            .chart-container, .table-container, .extra-container {{
                flex: 0;
                padding: 10px;
                height: 100%;
            }}
            .table-container {{
                text-align: center;
                margin: auto;
                flex: 1; 
                min-width: 300px; 
            }}
            table {{
                width: 60%;
                border-collapse: collapse;
            }}
            th, td {{
                border: 1px solid black;
                padding: 8px;
                text-align: center;
            }}
            th {{
                background-color: #f2f2f2;
            }}
            img {{
                width: 100%;
                height: auto;
                min-width: 750px; /* Set a minimum width */
                min-height: 750px; 
                object-fit: contain;
            }}
            .wide-table {{
                width: 100%;
                table-layout: fixed;
            }}

            .wide-table th, .wide-table td {{
                padding: 8px;
                width: 40%;
                min-width: 100px;
                white-space: nowrap;
            }}
            
        </style>
    </head>
    <body>
        <h1>{current_month} Exports</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Exports Overview</h2>
                    <img src="chart1" alt="SBM Exports overview">
                </td>
            </tr>
        </table>
        <h1>{current_month} Exports by Destinations</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Destinations</h2>
                    <img src="chart2" alt="Board Crush Chart">
                </td>
                <!-- Table Section -->
                <td class="table-container">
                    {html_byregions}  <!-- Insert DataFrame as an HTML table -->
                </td>
            </tr>
        </table>
        <h1>Top 10 Destinations</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <h2>Main Destinations</h2>
                    <img src="chart3" alt="top 10">
                </td>
                <!-- Table Section -->
                <td class="table-container">
                    <h2>Accumulated 2025 exports of main destinations</h2>
                    {top_10_countries}  <!-- Insert DataFrame as an HTML table -->
                </td>
            </tr>
        </table>
        <h1>Top 10 players</h1>
        <table class="layout">
            <tr>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <img src="chart4" alt="Top companies">
                </td>
                <!-- New Container Section (before the chart) -->
                <td class="extra-container">
                    <img src="chart5" alt="Top companies">
                </td>
            </tr>
        </table>
    </body>
    </html>
    """

    adress_test=['florian.girardi-ext@ldc.com','michelle.lambrechts@ldc.com',"SANTIAGO.CARBAJAL@ldc.com"]

    moi=['florian.girardi-ext@ldc.com']

    # Send email with embedded chart and table
    LDCDataAccessLayerPy.mail.mail_send(
        to=moi,
        subject=f'SBM Lineups  eqwq {datetime.now().strftime("%d-%m")}',
        from_addr="florian.girardi-ext@ldc.com",
        body=html_content,
        mime_type="html",
        html_images={"chart1": overview,"chart2":bar,'chart3':top,'chart4':fig_shippers,'chart5':fig_coordinators}  # Embedding the chart image
    )



